In [42]:
import numpy as np
import pandas as pd
import scipy
import scipy.stats
import matplotlib as mpl   
import matplotlib.pyplot as plt
from tqdm import tqdm
import seaborn as sns
import arviz as az
import pymc as pm
%matplotlib inline

In [43]:
import fastf1
import pandas as pd
import numpy as np
from datetime import datetime
import os

# Initialize cache directory for FastF1 API
CACHE_PATH = './data/f1_cache'
os.makedirs('./data', exist_ok=True)
os.makedirs(CACHE_PATH, exist_ok=True)
print(f"✅ Cache directory ready: {CACHE_PATH}")

fastf1.Cache.enable_cache(CACHE_PATH)

# Circuit classification by characteristics
CIRCUIT_CATEGORIES = {
    'Bahrain': 'high_speed', 'Saudi Arabia': 'high_speed', 'Australia': 'balanced',
    'Japan': 'technical', 'China': 'high_speed', 'Miami': 'high_speed',
    'Emilia Romagna': 'technical', 'Monaco': 'technical', 'Canada': 'balanced',
    'Spain': 'balanced', 'Austria': 'technical', 'Great Britain': 'balanced',
    'Hungary': 'technical', 'Belgium': 'high_speed', 'Netherlands': 'technical',
    'Italy': 'high_speed', 'Azerbaijan': 'high_speed', 'Singapore': 'technical',
    'United States': 'balanced', 'Mexico': 'balanced', 'Brazil': 'balanced',
    'Las Vegas': 'high_speed', 'Qatar': 'high_speed', 'Abu Dhabi': 'balanced',
    'Pre-Season': 'balanced',
}


def verify_cached_session(season_year, event_name, session_identifier):
    """
    Check if specific session data exists in local cache
    
    Args:
        season_year: Championship year
        event_name: Grand Prix name
        session_identifier: Session type code
    
    Returns:
        Boolean indicating cache availability
    """
    if not os.path.exists(CACHE_PATH):
        return False
    
    normalized_event = event_name.replace(' ', '_')
    
    for root_dir, subdirs, files in os.walk(CACHE_PATH):
        for subdir in subdirs:
            if normalized_event in subdir and str(season_year) in subdir:
                return True
    return False


✅ Cache directory ready: ./f1_cache


In [44]:
def load_single_event_data(season_year, event_name, event_round, event_date):
    """
    Retrieve race and qualifying session results for a single event
    
    Args:
        season_year: Year of the championship
        event_name: Name of the Grand Prix
        event_round: Round number in calendar
        event_date: Date of the event
    
    Returns:
        DataFrame with combined race and qualifying data
    """
    try:
        race_cached = verify_cached_session(season_year, event_name, 'R')
        quali_cached = verify_cached_session(season_year, event_name, 'Q')
        
        cache_status = "Using cached data" if (race_cached and quali_cached) else "Downloading data..."
        print(f"{'✓' if race_cached and quali_cached else '⬇'} Round {event_round}: {event_name} ({season_year}) - {cache_status}")
        
        # Load qualifying session
        quali_position_map = {}
        try:
            quali_obj = fastf1.get_session(season_year, event_name, 'Q')
            quali_obj.load()
            quali_results_df = quali_obj.results
            
            quali_position_map = dict(zip(
                quali_results_df['Abbreviation'],
                quali_results_df['Position']
            ))
            
        except Exception as quali_error:
            print(f"  ⚠️ Qualifying data unavailable: {quali_error}")
        
        # Load race session
        race_obj = fastf1.get_session(season_year, event_name, 'R')
        race_obj.load()
        race_results_df = race_obj.results
        
        # Build structured race dataset
        structured_data = pd.DataFrame({
            'Round': event_round,
            'Race': event_name,
            'Date': event_date,
            'TrackType': CIRCUIT_CATEGORIES.get(event_name.split()[0], 'balanced'),  # Use first word
            'Driver': race_results_df['BroadcastName'],
            'DriverCode': race_results_df['Abbreviation'],
            'DriverNumber': race_results_df['DriverNumber'],
            'Team': race_results_df['TeamName'],
            'GridPosition': race_results_df['GridPosition'],
            'Position': race_results_df['Position'],
            'Points': race_results_df['Points'],
            'Status': race_results_df['Status'],
            'Time': race_results_df['Time'],
        })
        
        # Map qualifying positions to race data
        structured_data['QualifyingPosition'] = structured_data['DriverCode'].map(quali_position_map)
        
        # Use grid position as fallback for missing qualifying data
        structured_data['QualifyingPosition'].fillna(structured_data['GridPosition'], inplace=True)
        
        return structured_data
        
    except Exception as load_error:
        print(f"❌ Failed to load {event_name}: {load_error}")
        return None

In [45]:
def aggregate_season_data(season_list=[2025], force_reload=False):
    """
    Aggregate race results across multiple seasons
    
    Args:
        season_list: List of championship years to process
        force_reload: Force data refresh ignoring existing CSV
    
    Returns:
        Combined DataFrame of all race results
    """
    output_csv = './data/f1_multi_season_results.csv'
    
    if os.path.exists(output_csv) and not force_reload:
        print(f"✅ Loading existing dataset: {output_csv}")
        combined_df = pd.read_csv(output_csv)
        print(f"Loaded {len(combined_df)} entries from {combined_df['Round'].nunique()} events")
        return combined_df
    
    print(f"Initiating data collection for seasons: {season_list}...")
    collected_events = []
    
    for year in season_list:
        print(f"\n{'='*60}")
        print(f"PROCESSING {year} CHAMPIONSHIP")
        print(f"{'='*60}")
        
        try:
            calendar = fastf1.get_event_schedule(year)
            
            completed_events = calendar[calendar['EventDate'] < pd.Timestamp.now()]
            
            for idx, event_row in completed_events.iterrows():
                round_idx = event_row['RoundNumber']
                gp_name = event_row['EventName']
                gp_date = event_row['EventDate'].strftime('%Y-%m-%d')
                
                event_df = load_single_event_data(year, gp_name, round_idx, gp_date)
                
                if event_df is not None:
                    event_df['Season'] = year
                    collected_events.append(event_df)
                    
        except Exception as schedule_error:
            print(f"❌ Schedule retrieval failed for {year}: {schedule_error}")
    
    if collected_events:
        combined_df = pd.concat(collected_events, ignore_index=True)
        
        combined_df.sort_values(['Season', 'Round'], inplace=True)
        combined_df['GlobalRound'] = combined_df.groupby(['Season', 'Round']).ngroup() + 1
        
        combined_df.to_csv(output_csv, index=False)
        
        print(f"\n{'='*60}")
        print(f"COLLECTION COMPLETED")
        print(f"{'='*60}")
        print(f"✓ Total entries: {len(combined_df)}")
        print(f"✓ Seasons included: {combined_df['Season'].unique()}")
        print(f"✓ Events collected: {combined_df.groupby(['Season', 'Round']).ngroups}")
        print(f"✓ Unique drivers: {combined_df['DriverCode'].nunique()}")
        
        print("\nSeason breakdown:")
        print(combined_df['Season'].value_counts().sort_index())
        
        return combined_df
    else:
        print("❌ Data collection yielded no results")
        return None


def build_driver_summary(season_df):
    """
    Extract driver standings and basic information
    
    Args:
        season_df: Full season race results
    
    Returns:
        DataFrame with driver standings
    """
    final_round = season_df[season_df['Round'] == season_df['Round'].max()]
    
    driver_summary = final_round[['Driver', 'DriverCode', 'Team']].copy()
    driver_summary.drop_duplicates(subset=['Driver'], inplace=True)
    
    points_totals = season_df.groupby('Driver')['Points'].sum().reset_index()
    points_totals.columns = ['Driver', 'TotalPoints']
    
    driver_summary = driver_summary.merge(points_totals, on='Driver')
    
    driver_summary.sort_values('TotalPoints', ascending=False, inplace=True)
    driver_summary.reset_index(drop=True, inplace=True)
    driver_summary['CurrentRank'] = range(1, len(driver_summary) + 1)
    
    driver_summary.to_csv('./data/drivers_info.csv', index=False)
    print("\n✅ Driver standings saved to drivers_info.csv")
    
    return driver_summary

In [46]:
def build_team_summary(season_df):
    """
    Calculate constructor standings with tier classification
    
    Args:
        season_df: Full season race results
    
    Returns:
        DataFrame with team standings and tiers
    """
    constructor_points = season_df.groupby('Team')['Points'].sum().reset_index()
    constructor_points.columns = ['Team', 'TotalPoints']
    constructor_points.sort_values('TotalPoints', ascending=False, inplace=True)
    constructor_points.reset_index(drop=True, inplace=True)
    constructor_points['CurrentRank'] = range(1, len(constructor_points) + 1)
    
    def classify_team_tier(rank_pos):
        if rank_pos <= 3:
            return 0
        elif rank_pos <= 5:
            return 1
        else:
            return 2
    
    constructor_points['Tier'] = constructor_points['CurrentRank'].apply(classify_team_tier)
    
    constructor_points.to_csv('./data/teams_info.csv', index=False)
    print("✅ Constructor standings saved to teams_info.csv")
    
    return constructor_points


def compute_driver_metrics(season_df):
    """
    Generate comprehensive performance metrics per driver
    
    Args:
        season_df: Full season race results
    
    Returns:
        DataFrame with detailed driver statistics
    """
    if 'TrackType' not in season_df.columns:
        print("⚠️ TrackType column missing, adding from mapping...")
        season_df['TrackType'] = season_df['Race'].map(CIRCUIT_CATEGORIES)
    
    driver_list = season_df['Driver'].unique()
    metric_records = []
    
    for current_driver in driver_list:
        driver_races = season_df[season_df['Driver'] == current_driver].copy()
        
        latest_team = driver_races['Team'].iloc[-1]
        cumulative_points = driver_races['Points'].sum()
        
        race_count = len(driver_races)
        completed_count = len(driver_races[driver_races['Status'] == 'Finished'])
        retirement_ratio = 1 - (completed_count / race_count) if race_count > 0 else 0
        
        completed_races = driver_races[driver_races['Status'] == 'Finished']
        mean_finish = completed_races['Position'].mean() if len(completed_races) > 0 else 20
        
        last_five = driver_races.tail(5)
        last_five_complete = last_five[last_five['Status'] == 'Finished']
        recent_mean = last_five_complete['Position'].mean() if len(last_five_complete) > 0 else mean_finish
        
        circuit_stats = {}
        for circuit_cat in ['high_speed', 'balanced', 'technical']:
            cat_races = driver_races[driver_races['TrackType'] == circuit_cat]
            cat_complete = cat_races[cat_races['Status'] == 'Finished']
            circuit_stats[f'{circuit_cat}_avg'] = cat_complete['Position'].mean() if len(cat_complete) > 0 else mean_finish
            circuit_stats[f'{circuit_cat}_races'] = len(cat_complete)
        
        metric_records.append({
            'Driver': current_driver,
            'Team': latest_team,
            'TotalPoints': cumulative_points,
            'TotalRaces': race_count,
            'FinishedRaces': completed_count,
            'DNFRate': retirement_ratio,
            'AvgPosition': mean_finish,
            'Recent5Avg': recent_mean,
            'HighSpeedAvg': circuit_stats['high_speed_avg'],
            'HighSpeedRaces': circuit_stats['high_speed_races'],
            'BalancedAvg': circuit_stats['balanced_avg'],
            'BalancedRaces': circuit_stats['balanced_races'],
            'TechnicalAvg': circuit_stats['technical_avg'],
            'TechnicalRaces': circuit_stats['technical_races'],
        })
    
    metrics_df = pd.DataFrame(metric_records)
    metrics_df.sort_values('TotalPoints', ascending=False, inplace=True)
    metrics_df.reset_index(drop=True, inplace=True)
    
    metrics_df.to_csv('./data/driver_features.csv', index=False)
    print("✅ Driver metrics saved to driver_features.csv")
    
    return metrics_df

In [47]:
def aggregate_manual_calendar(force_reload=False):
    """
    Load race data using hardcoded event calendar
    Bypasses schedule API for reliability
    
    Args:
        force_reload: Force refresh ignoring cached CSV
    
    Returns:
        Combined DataFrame of race results
    """
    output_csv = './data/f1_multi_season_results.csv'
    
    if os.path.exists(output_csv) and not force_reload:
        print(f"✅ Loading cached dataset: {output_csv}")
        cached_df = pd.read_csv(output_csv)
        
        if 'TrackType' not in cached_df.columns:
            print("⚠️ Enriching dataset with TrackType...")
            cached_df['TrackType'] = cached_df['Race'].map(CIRCUIT_CATEGORIES)
            cached_df.to_csv(output_csv, index=False)
            print("✅ TrackType column added")
        
        print(f"Dataset contains {len(cached_df)} records")
        return cached_df
    
    calendar_2024 = [
        ('Bahrain', 1, '2024-03-02'),
        ('Saudi Arabia', 2, '2024-03-09'),
        ('Australia', 3, '2024-03-24'),
        ('Japan', 4, '2024-04-07'),
        ('China', 5, '2024-04-21'),
        ('Miami', 6, '2024-05-05'),
        ('Emilia Romagna', 7, '2024-05-19'),
        ('Monaco', 8, '2024-05-26'),
        ('Canada', 9, '2024-06-09'),
        ('Spain', 10, '2024-06-23'),
        ('Austria', 11, '2024-06-30'),
        ('Great Britain', 12, '2024-07-07'),
        ('Hungary', 13, '2024-07-21'),
        ('Belgium', 14, '2024-07-28'),
        ('Netherlands', 15, '2024-08-25'),
        ('Italy', 16, '2024-09-01'),
        ('Azerbaijan', 17, '2024-09-15'),
        ('Singapore', 18, '2024-09-22'),
        ('United States', 19, '2024-10-20'),
        ('Mexico', 20, '2024-10-27'),
        ('Brazil', 21, '2024-11-03'),
        ('Las Vegas', 22, '2024-11-23'),
        ('Qatar', 23, '2024-12-01'),
        ('Abu Dhabi', 24, '2024-12-08'),
    ]
    
    calendar_2025 = [
        ('Australia', 1, '2025-03-16'),
        ('China', 2, '2025-03-23'),
        ('Japan', 3, '2025-04-06'),
        ('Bahrain', 4, '2025-04-13'),
        ('Saudi Arabia', 5, '2025-04-20'),
        ('Miami', 6, '2025-05-04'),
        ('Emilia Romagna', 7, '2025-05-18'),
        ('Monaco', 8, '2025-05-25'),
        ('Spain', 9, '2025-06-01'),
        ('Canada', 10, '2025-06-15'),
        ('Austria', 11, '2025-06-29'),
        ('Great Britain', 12, '2025-07-06'),
        ('Belgium', 13, '2025-07-27'),
        ('Hungary', 14, '2025-08-03'),
        ('Netherlands', 15, '2025-08-31'),
        ('Italy', 16, '2025-09-07'),
        ('Azerbaijan', 17, '2025-09-21'),
        ('Singapore', 18, '2025-10-05'),
        ('United States', 19, '2025-10-19'),
        ('Mexico', 20, '2025-10-26'),
        ('Brazil', 21, '2025-11-09'),
        ('Las Vegas', 22, '2025-11-22'),
        ('Qatar', 23, '2025-11-30'),
    ]
    
    all_event_data = []
    
    print(f"\n{'='*60}")
    print(f"LOADING 2025 SEASON")
    print(f"{'='*60}")
    for gp_name, round_num, date_str in calendar_2025:
        event_data = load_single_event_data(2025, gp_name, round_num, date_str)
        if event_data is not None:
            event_data['Season'] = 2025
            all_event_data.append(event_data)
    
    if all_event_data:
        full_dataset = pd.concat(all_event_data, ignore_index=True)
        full_dataset.sort_values(['Season', 'Round'], inplace=True)
        full_dataset['GlobalRound'] = full_dataset.groupby(['Season', 'Round']).ngroup() + 1
        
        if 'TrackType' not in full_dataset.columns:
            full_dataset['TrackType'] = full_dataset['Race'].map(CIRCUIT_CATEGORIES)
            print("✅ TrackType column integrated")
        
        full_dataset.to_csv(output_csv, index=False)
        
        print(f"\n{'='*60}")
        print(f"COLLECTION COMPLETE")
        print(f"{'='*60}")
        print(f"✓ Total entries: {len(full_dataset)}")
        print(f"✓ Events processed: {full_dataset.groupby(['Season', 'Round']).ngroups}")
        print(f"✓ TrackType included: {'Yes' if 'TrackType' in full_dataset.columns else 'No'}")
        
        if 'TrackType' in full_dataset.columns:
            print(f"\nCircuit type distribution:")
            print(full_dataset.groupby('TrackType')['Round'].nunique())
        
        return full_dataset
    else:
        print("❌ No data collected")
        return None

In [48]:
# Execute data collection pipeline
full_season = aggregate_season_data(
    season_list=[2025],
    force_reload=False
)

if full_season is not None:
    teams_info = build_team_summary(full_season)
    drivers_info = build_driver_summary(full_season)
    driver_features = compute_driver_metrics(full_season)
    
    print("\n✅ Data pipeline completed successfully")
    print(f"   - Total records: {len(full_season)}")
    print(f"   - Seasons: {full_season['Season'].unique()}")
    print(f"   - QualifyingPosition available: {'QualifyingPosition' in full_season.columns}")
else:
    print("❌ Data collection failed")

# Display driver standings
driver_info = build_driver_summary(full_season)
print("\n  TOP 10 DRIVERS:")
print(driver_info.head(10))

print("\n" + "="*50)
team_info = build_team_summary(full_season)
print("\n CONSTRUCTOR STANDINGS:")
print(team_info)

print("\n" + "="*50)
driver_features = compute_driver_metrics(full_season)
print("\n TOP 10 DRIVER METRICS:")
print(driver_features[['Driver', 'TotalPoints', 'AvgPosition', 'Recent5Avg', 
                       'HighSpeedAvg', 'BalancedAvg', 'DNFRate']].head(10))

Initiating data collection for seasons: [2025]...

PROCESSING 2025 CHAMPIONSHIP


events      WARNING 	Correcting user input 'Pre-Season Testing' to 'Singapore Grand Prix'
core           INFO 	Loading data for Singapore Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


⬇ Round 0: Pre-Season Testing (2025) - Downloading data...


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['63', '1', '81', '12', '4', '44', '16', '6', '87', '14', '27', '30', '22', '5', '18', '43', '31', '10', '23', '55']
events      WARNING 	Correcting user input 'Pre-Season Testing' to 'Singapore Grand Prix'
core           INFO 	Loading data for Singapore Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Us

✓ Round 1: Australian Grand Prix (2025) - Using cached data


...

req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '1', '55', '63', '81', '30', '14', '6', '16', '10', '27', '18', '31', '87', '43', '23', '12', '5', '22', '44']
core           INFO 	Loading data for Las Vegas Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_cou

⬇ Round 23: Qatar Grand Prix (2025) - Downloading data...


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for timing_app_data. Loading data...
_api           INFO 	Fetching timing app data...
req            INFO 	Data has been written to


COLLECTION COMPLETED
✓ Total entries: 479
✓ Seasons included: [2025]
✓ Events collected: 24
✓ Unique drivers: 21

Season breakdown:
Season
2025    479
Name: count, dtype: int64
✅ Constructor standings saved to teams_info.csv

✅ Driver standings saved to drivers_info.csv
✅ Driver metrics saved to driver_features.csv

✅ Data pipeline completed successfully
   - Total records: 479
   - Seasons: [2025]
   - QualifyingPosition available: True

✅ Driver standings saved to drivers_info.csv

  TOP 10 DRIVERS:
         Driver DriverCode             Team  TotalPoints  CurrentRank
0      L NORRIS        NOR          McLaren        394.0            1
1  M VERSTAPPEN        VER  Red Bull Racing        382.0            2
2     O PIASTRI        PIA          McLaren        375.0            3
3     G RUSSELL        RUS         Mercedes        304.0            4
4     C LECLERC        LEC          Ferrari        221.0            5
5    L HAMILTON        HAM          Ferrari        135.0            6
6 

/var/folders/h5/pz9lkn8j6_bdxpczx2pqr0fr0000gn/T/ipykernel_16650/3273122541.py:62: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  structured_data['QualifyingPosition'].fillna(structured_data['GridPosition'], inplace=True)
